In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

!pip install transformers torch datasets scikit-learn -q

In [ ]:
print('Expected local files:')
print('  1. daraz_dataset.csv')
print('  2. semeval_normalization.py')

Upload TWO files:
  1. daraz_dataset.csv
  2. semeval_normalization.py


In [ ]:
import importlib, semeval_normalization
importlib.reload(semeval_normalization)
from semeval_normalization import normalize_semeval_spans, validate_normalization
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

dataset = load_dataset('tomaarsen/setfit-absa-semeval-laptops')
train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

train_df = train_df[train_df['label'] != 'conflict'].reset_index(drop=True)
test_df  = test_df[test_df['label']  != 'conflict'].reset_index(drop=True)

unique_sents = train_df['text'].unique()
train_sents, val_sents = train_test_split(unique_sents, test_size=0.2, random_state=42)

final_train = train_df[train_df['text'].isin(train_sents)].reset_index(drop=True)
final_val   = train_df[train_df['text'].isin(val_sents)].reset_index(drop=True)

final_train = final_train.drop_duplicates(subset=['text','span','label']).reset_index(drop=True)
final_val   = final_val.drop_duplicates(subset=['text','span','label']).reset_index(drop=True)
final_train = final_train[final_train['text'].str.split().str.len() >= 3].reset_index(drop=True)

print(f'Train: {len(final_train)} rows | Val: {len(final_val)} rows')
print(f'Raw unique spans: {final_train["span"].str.lower().nunique()}')

original_train = final_train.copy()
final_train = normalize_semeval_spans(final_train)
final_val   = normalize_semeval_spans(final_val)

validate_normalization(original_train, final_train)
print('\nNormalization applied.')

Train: 1799 rows | Val: 465 rows
Raw unique spans: 806
Original unique spans:    806
Normalized unique spans:  24
Spans using final fallback ('features'): 461
  Examples: ['9 punds', 'abilitiy', 'accessories', 'acer arcade', 'adobe creative apps', 'adobe creative suite', 'aero', 'aesthetics', 'affordability', 'agents']

Final category distribution:
span
features            863
software            158
screen               83
battery life         82
customer service     71
keyboard             71
performance          65
price                60
design               50
build quality        42
warranty             40
touchscreen          32
connectivity         29
sound quality        24
charger              23
processor            19
display              17
heating              15
camera               14
laptop               11
speaker              11
delivery             10
cable                 5
charging speed        4
Name: count, dtype: int64

Normalization applied.


In [ ]:
import torch

label2id = {'positive': 0, 'negative': 1, 'neutral': 2}
id2label  = {0: 'positive', 1: 'negative', 2: 'neutral'}

final_train['label_id'] = final_train['label'].map(label2id)
final_val['label_id']   = final_val['label'].map(label2id)

total  = len(final_train)
counts = final_train['label'].value_counts()
weights = [
    total / (3 * counts['positive']),
    total / (3 * counts['negative']),
    total / (3 * counts['neutral'])
]
class_weights = torch.tensor(weights, dtype=torch.float)
print(f'Class weights: {class_weights}')

Class weights: tensor([0.7922, 0.8703, 1.6988])


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['span']),
            str(row['txt'] if 'txt' in row else row['text']),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc['token_type_ids'].squeeze(),
            'label': torch.tensor(row['label_id'], dtype=torch.long)
        }

BATCH_SIZE = 16
train_dataset = ABSADataset(final_train, tokenizer)
val_dataset   = ABSADataset(final_val,   tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to(device)

EPOCHS      = 4
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(params, lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

print(f'Total training steps: {total_steps} | Warmup steps: {warmup_steps}')

Total training steps: 452 | Warmup steps: 45


In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        out = model(input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    token_type_ids=batch['token_type_ids'].to(device))
        labels = batch['label'].to(device)
        loss   = criterion(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(all_labels, all_preds, average='macro')

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device),
                        token_type_ids=batch['token_type_ids'].to(device))
            labels = batch['label'].to(device)
            loss   = criterion(out.logits, labels)
            total_loss += loss.item()
            all_preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels

print('Training and evaluation functions defined.')

Training and evaluation functions defined.


In [ ]:
SAVE_NAME   = MODEL_NAME.replace('/', '_')
SAVE_PATH   = f'checkpoints/best_{SAVE_NAME}.pt'
best_val_f1 = 0
best_epoch  = 0

print(f'Training {MODEL_NAME}...\n')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1       = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    vl_loss, vl_f1, _, _ = eval_epoch(model, val_loader, criterion, device)
    print(f'Epoch {epoch}/{EPOCHS}  |  Train Loss: {tr_loss:.4f}  Train F1: {tr_f1:.4f}  |  Val Loss: {vl_loss:.4f}  Val F1: {vl_f1:.4f}')
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  --> Saved best model (epoch {epoch})')

print(f'\nBest: Epoch {best_epoch}  |  Val Macro F1 = {best_val_f1:.4f}')

Training bert-base-uncased...

Epoch 1/4  |  Train Loss: 0.9655  Train F1: 0.5207  |  Val Loss: 0.7829  Val F1: 0.6591
  --> Saved best model (epoch 1)
Epoch 2/4  |  Train Loss: 0.6649  Train F1: 0.7289  |  Val Loss: 0.7281  Val F1: 0.6738
  --> Saved best model (epoch 2)
Epoch 3/4  |  Train Loss: 0.4927  Train F1: 0.8058  |  Val Loss: 0.7450  Val F1: 0.6787
  --> Saved best model (epoch 3)
Epoch 4/4  |  Train Loss: 0.4031  Train F1: 0.8556  |  Val Loss: 0.7515  Val F1: 0.6838
  --> Saved best model (epoch 4)

Best: Epoch 4  |  Val Macro F1 = 0.6838


In [ ]:
model.load_state_dict(torch.load(SAVE_PATH))
_, _, semeval_preds, semeval_labels = eval_epoch(model, val_loader, criterion, device)

print(f"\n{'='*55}")
print(f'SEMEVAL VAL RESULTS — {MODEL_NAME}')
print(f"{'='*55}")
print(classification_report(semeval_labels, semeval_preds,
    target_names=['positive','negative','neutral'], digits=4))

daraz_df = pd.read_csv('daraz_dataset.csv')
daraz_df['label_id'] = daraz_df['label'].map(label2id)
daraz_df = daraz_df.dropna(subset=['label_id']).reset_index(drop=True)
daraz_df['label_id'] = daraz_df['label_id'].astype(int)
daraz_dataset_eval = ABSADataset(daraz_df, tokenizer)
daraz_loader_eval  = DataLoader(daraz_dataset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

_, _, daraz_preds, daraz_labels = eval_epoch(model, daraz_loader_eval, criterion, device)
print(f"\n{'='*55}")
print(f'DARAZ ZERO-SHOT RESULTS — {MODEL_NAME}')
print(f"{'='*55}")
print(classification_report(daraz_labels, daraz_preds,
    target_names=['positive','negative','neutral'], digits=4))


SEMEVAL VAL RESULTS — bert-base-uncased
              precision    recall  f1-score   support

    positive     0.7854    0.8190    0.8019       210
    negative     0.7727    0.7778    0.7752       153
     neutral     0.5000    0.4510    0.4742       102

    accuracy                         0.7247       465
   macro avg     0.6860    0.6826    0.6838       465
weighted avg     0.7186    0.7247    0.7212       465


DARAZ ZERO-SHOT RESULTS — bert-base-uncased
              precision    recall  f1-score   support

    positive     0.5029    0.4293    0.4632       410
    negative     0.6600    0.1119    0.1913       295
     neutral     0.1018    0.3285    0.1554       137

    accuracy                         0.3017       842
   macro avg     0.4216    0.2899    0.2700       842
weighted avg     0.4927    0.3017    0.3178       842


In [ ]:
MBERT_NAME      = 'bert-base-multilingual-cased'
mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)

mbert_train_dataset = ABSADataset(final_train, mbert_tokenizer)
mbert_val_dataset   = ABSADataset(final_val,   mbert_tokenizer)
mbert_train_loader  = DataLoader(mbert_train_dataset, batch_size=16, shuffle=True,  num_workers=2)
mbert_val_loader    = DataLoader(mbert_val_dataset,   batch_size=16, shuffle=False, num_workers=2)

print(f'mBERT tokenizer loaded: {MBERT_NAME}')
print(f'Train batches: {len(mbert_train_loader)} | Val batches: {len(mbert_val_loader)}')

mBERT tokenizer loaded: bert-base-multilingual-cased
Train batches: 113 | Val batches: 30


In [ ]:
mbert_model = AutoModelForSequenceClassification.from_pretrained(
    MBERT_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to(device)

EPOCHS       = 4
total_steps  = len(mbert_train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

no_decay = ['bias', 'LayerNorm.weight']
mbert_params = [
    {'params': [p for n,p in mbert_model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n,p in mbert_model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
mbert_optimizer = AdamW(mbert_params, lr=2e-5)
mbert_scheduler = get_linear_schedule_with_warmup(mbert_optimizer, warmup_steps, total_steps)
mbert_criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
MBERT_SAVE = 'checkpoints/best_mbert.pt'

print(f'Total steps: {total_steps} | Warmup: {warmup_steps}')

Total steps: 452 | Warmup: 45


In [ ]:
best_mbert_f1    = 0
best_mbert_epoch = 0

print(f'Training {MBERT_NAME}...\n')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1       = train_epoch(mbert_model, mbert_train_loader, mbert_optimizer, mbert_scheduler, mbert_criterion, device)
    vl_loss, vl_f1, _, _ = eval_epoch(mbert_model, mbert_val_loader, mbert_criterion, device)
    print(f'Epoch {epoch}/{EPOCHS}  |  Train Loss: {tr_loss:.4f}  Train F1: {tr_f1:.4f}  |  Val Loss: {vl_loss:.4f}  Val F1: {vl_f1:.4f}')
    if vl_f1 > best_mbert_f1:
        best_mbert_f1    = vl_f1
        best_mbert_epoch = epoch
        torch.save(mbert_model.state_dict(), MBERT_SAVE)
        print(f'  --> Saved best model (epoch {epoch})')

print(f'\nBest: Epoch {best_mbert_epoch}  |  Val Macro F1 = {best_mbert_f1:.4f}')

Training bert-base-multilingual-cased...

Epoch 1/4  |  Train Loss: 0.9861  Train F1: 0.4877  |  Val Loss: 0.8070  Val F1: 0.5986
  --> Saved best model (epoch 1)
Epoch 2/4  |  Train Loss: 0.6846  Train F1: 0.7300  |  Val Loss: 0.7463  Val F1: 0.6332
  --> Saved best model (epoch 2)
Epoch 3/4  |  Train Loss: 0.4952  Train F1: 0.8191  |  Val Loss: 0.8459  Val F1: 0.6360
  --> Saved best model (epoch 3)
Epoch 4/4  |  Train Loss: 0.3897  Train F1: 0.8546  |  Val Loss: 0.8387  Val F1: 0.6410
  --> Saved best model (epoch 4)

Best: Epoch 4  |  Val Macro F1 = 0.6410


In [ ]:
mbert_model.load_state_dict(torch.load(MBERT_SAVE))

_, _, semeval_preds_m, semeval_labels_m = eval_epoch(mbert_model, mbert_val_loader, mbert_criterion, device)
print(f"\n{'='*55}")
print(f'SEMEVAL VAL RESULTS — {MBERT_NAME}')
print(f"{'='*55}")
print(classification_report(semeval_labels_m, semeval_preds_m,
    target_names=['positive','negative','neutral'], digits=4))

mbert_daraz_dataset = ABSADataset(daraz_df, mbert_tokenizer)
mbert_daraz_loader  = DataLoader(mbert_daraz_dataset, batch_size=16, shuffle=False, num_workers=2)

_, _, daraz_preds_m, daraz_labels_m = eval_epoch(mbert_model, mbert_daraz_loader, mbert_criterion, device)
print(f"\n{'='*55}")
print(f'DARAZ ZERO-SHOT RESULTS — {MBERT_NAME}')
print(f"{'='*55}")
print(classification_report(daraz_labels_m, daraz_preds_m,
    target_names=['positive','negative','neutral'], digits=4))

import json
mbert_results = {
    'model': MBERT_NAME,
    'semeval_val':    classification_report(semeval_labels_m, semeval_preds_m,
                        target_names=['positive','negative','neutral'], digits=4, output_dict=True),
    'daraz_zeroshot': classification_report(daraz_labels_m, daraz_preds_m,
                        target_names=['positive','negative','neutral'], digits=4, output_dict=True)
}
with open('checkpoints/results_mbert.json', 'w') as f:
    json.dump(mbert_results, f, indent=2)

print(f'\nKey numbers:')
print(f'  SemEval Val  Macro F1 : {mbert_results["semeval_val"]["macro avg"]["f1-score"]:.4f}')
print(f'  Daraz Z-Shot Macro F1 : {mbert_results["daraz_zeroshot"]["macro avg"]["f1-score"]:.4f}')
print('Results saved to checkpoints/.')


SEMEVAL VAL RESULTS — bert-base-multilingual-cased
              precision    recall  f1-score   support

    positive     0.7451    0.7238    0.7343       210
    negative     0.6943    0.7124    0.7032       153
     neutral     0.4808    0.4902    0.4854       102

    accuracy                         0.6688       465
   macro avg     0.6400    0.6421    0.6410       465
weighted avg     0.6704    0.6688    0.6695       465


DARAZ ZERO-SHOT RESULTS — bert-base-multilingual-cased
              precision    recall  f1-score   support

    positive     0.6190    0.2220    0.3268       410
    negative     0.4083    0.7017    0.5162       295
     neutral     0.1915    0.2628    0.2215       137

    accuracy                         0.3967       842
   macro avg     0.4063    0.3955    0.3548       842
weighted avg     0.4756    0.3967    0.3760       842


Key numbers:
  SemEval Val  Macro F1 : 0.6410
  Daraz Z-Shot Macro F1 : 0.3548
Results saved to Drive.
